In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("hf://datasets/sharjeelyunus/github-issues-dataset/github_issues_dataset.parquet")

c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df.drop(["id", "repo"], axis=1, inplace=True)

## Normalize labels column

In [4]:
import re
import pandas as pd

def clean_labels(raw_labels):
    if pd.isna(raw_labels):
        return ["unknown"]
        
    noise_labels = {"p1", "p2", "p3", "p4", "p5"}
    parts = [l.strip().lower() for l in str(raw_labels).split(",")]
    meaningful = [
        l for l in parts
        if re.search(r"[a-z]", l) and l not in noise_labels
    ]
    return meaningful if meaningful else ["unknown"]

df["labels-list"] = df["labels"].apply(clean_labels)


In [5]:
from collections import Counter

all_labels = [l for row in df["labels-list"] for l in row]
counts = Counter(all_labels)

In [6]:
import random
import json

with open('map.json', 'r', encoding='utf-8') as f:
    alias_to_group = json.load(f)

def process_labels(labels):
    mapped = []
    seen = set()
    
    for label in labels:
        group = alias_to_group.get(label.lower(), "unknown")
        
        # Consolidate 'other' to 'unknown'
        if group == "other":
            group = "unknown"
            
        if group not in seen:
            mapped.append(group)
            seen.add(group)
    
    # Mapped array resolution
    if len(mapped) > 1 and "unknown" in mapped:
        mapped.remove("unknown")
    final_mapped = mapped if mapped else ["unknown"]
        
    return final_mapped

df["labels-mapped"] = df["labels-list"].apply(process_labels)
df["labels-mapped"].explode().value_counts().head(20)


labels-mapped
bug                     43045
feature_request         31239
ui_ux                   28258
compatibility           17882
build_ci_cd             16969
question                15803
performance             11235
integration_api          9674
testing                  8577
documentation            7201
unknown                  7123
data_import_export       2581
refactor_maintenance     2574
configuration            1766
access_permissions        922
installation_setup        909
security                  732
duplicate_existing         98
Name: count, dtype: int64

### Removing entries with label "unkown" we will use them later for testing model

In [7]:
# Extract entries that are exclusively 'unknown'
df_unknown = df[df["labels-mapped"].apply(lambda x: x == ["unknown"])]

# Remove them from the main dataframe
df = df[df["labels-mapped"].apply(lambda x: x != ["unknown"])]

print(f"Unknown entries extracted: {len(df_unknown)}")
print(f"Main dataframe entries remaining: {len(df)}")

Unknown entries extracted: 7123
Main dataframe entries remaining: 106950


### Merging rare labels to aviod lon-tail problem.
- setup_config = installation_setup + configuration
- security_access = security + access_permissions
- maintenance_data = refactor_maintenance + data_import_export
- And drop "duplicate_existing"

In [8]:
label_merges = {
    "installation_setup": "setup_config",
    "configuration": "setup_config",
    "security": "security_access",
    "access_permissions": "security_access",
    "refactor_maintenance": "maintenance_data",
    "data_import_export": "maintenance_data"
}

def consolidate_rare_labels(labels_list):
    new_labels = set()
    for lbl in labels_list:
        if lbl == "duplicate_existing":
            continue
        new_labels.add(label_merges.get(lbl, lbl))
    return list(new_labels)

df["labels-mapped"] = df["labels-mapped"].apply(consolidate_rare_labels)

# Remove any rows that became empty because they only had the 'duplicate_existing' label
df = df[df["labels-mapped"].apply(lambda x: len(x) > 0)]

df["labels-mapped"].explode().value_counts().head(20)

labels-mapped
bug                 43045
feature_request     31239
ui_ux               28258
compatibility       17882
build_ci_cd         16969
question            15803
performance         11235
integration_api      9674
testing              8577
documentation        7201
maintenance_data     5123
setup_config         2667
security_access      1617
Name: count, dtype: int64

In [9]:
target_share = 0.83
total = sum(counts.values())
target = target_share * total

running = 0
n = 0
for _, c in counts.most_common():
    running += c
    n += 1
    if running >= target:
        break

print("n =", n)
print("coverage =", running / total)

n = 528
coverage = 0.8300270540492855


In [10]:
df.columns

Index(['title', 'body', 'labels', 'priority', 'severity', 'labels-list',
       'labels-mapped'],
      dtype='str')

In [11]:
df.to_csv('../dataset/dataset_13_labels.csv', columns=['title', 'body', 'labels-mapped'], index=False)

In [12]:
df.to_csv('../dataset/dataset_clean.csv', index=False)